# Laya V3.4 — Replace ALL Full-Attention Layers with Query-Routed Sparse Retrieval

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V34_AllFullAttention_QueryRoutedSparse_Colab.ipynb)

**Goal.** Evaluate a genuine, non-oracle replacement of **every pretrained ModernBERT full-attention layer**, preserving pretrained Q/K/V, rotary positional embeddings, Wo and *all sliding-attention layers*. The benchmark source is \`convaiinnovations/laya-typed-decisions\`.

**Evidence from V3.3** (teacher-informed ORACLES, NOT deployable): exact local 256 plus 8 oracle-selected remote keys achieved held-out layer-15 cosine **0.9916**; global oracle top-32 achieved **0.9771**; Gaussian-latent KV autoencoders did not provide acceptable fidelity. V3.4 implements the missing step: choose distant keys from query + block-level key representations **without ever computing the full T×T QK matrix at inference**.

## Two true replacement routes
1. \`local256_remote8\`: exact RoPE Q/K attention over up to **256 local keys** (127 left, self, 128 right), and up to **8 query-selected remote keys** found through coarse block routing.
2. \`global_top32\`: query-routed retrieval of up to **32 original keys**, selected from screened blocks. No mandatory local window; potentially less gathered data and potentially less fidelity.

Both modes are screened at **4 or 8 routing blocks per query**, with block size 16. A five-prototype block router (masked mean, first, middle, last, max-key-norm) scores \`[B,H,T,blocks,5]\`, chooses promising blocks, and then uses **exact pretrained RoPE QK scores only for shortlisted original keys**. A per-query unified softmax renormalizes over the retrieved keys. All positions/values remain original. The key selection is **non-differentiable heuristic routing** using frozen teacher Q/K; it is a genuine approximation, not V3.3's oracle.

**Important:** This is **sparse attention, not attention-free CeNN**. It does not train a new foundation model. It tests the pretrained layer replacement that V3.3's math suggests. The optimized native SDPA teacher may be faster: PyTorch advanced-index gathers and Python chunking may dominate. No speedup is assumed, and a lower numerical QK score count is NOT evidence of faster GPU execution.

## Scientific protocol
- A CPU/GPU mathematical preflight: exact selected-all parity with native attention, boolean/additive masks, short lengths, noisy masked padding, and exact pretrained layer call.
- On identical **case-disjoint probe samples**, independently select the highest-fidelity route for **each full-attention layer**. No audit data is used for route selection.
- Replace every detected \`full_attention\` layer in a *fresh pretrained* student, leaving \`sliding_attention\` unmodified; report cumulative downstream agreement, gold accuracy and JS after each installation.
- The report also compares single-layer local-output fidelity against frozen teacher inputs and measures native/replacement layer latency and full-model latency on matched batches.
- At the end, measure all-layers-replaced downstream accuracy, a separate held-out **audit** local fidelity and representative GPU timing.
- Save the winning per-layer strategy JSON; no pretrained model is silently overwritten or published. The official typed-decisions TEST split remains untouched.

### Acceptance targets (not predictions)
Held-out attention-output cosine ≥0.95 **per layer**, cumulative teacher decision agreement ≥0.95, no material gold-accuracy loss, and whole-model latency ≤native. If these fail, the model is **not ready**; the report keeps the exact failure values.

### Runtime
CUDA GPU required. The default experiment uses 512-token maximum sequences, 3 batches per layer for probe, 3 batches for held-out audit, and up to 120 audit decisions for end-to-end checks. All full-attention layers are replaced by default. Long sequences / many full layers can take minutes because sparse gathering is implemented in ordinary PyTorch, not a fused kernel.

**Additional diagnostic:** For each probe-selected route, compute teacher attention mass captured and actual top-8 key recall using explicit full teacher scores **only outside the inference path**. This helps distinguish bad candidate routing from sparse scoring errors. This diagnostic is never used for candidate selection.

## October 2026 FP16 fix

Routing block logits now explicitly compute in FP32 **with autocast disabled around einsum**, since CUDA autocast was producing FP16 tensors and `masked_fill(-1e9)` overflowed. Exact shortlisted Q/K score calculations also remain FP32. The preflight includes a 320-token FP16/BF16 autocast regression with 41 padded tokens. This is an implementation fix, not a fidelity improvement or GPU benchmark claim.


In [ ]:
#@title 1. Install exact training dependencies
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@main"


In [ ]:
#@title 2. Imports, frozen Laya teacher and experiment configuration
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import layer_kinds, MultiscaleCeNN
from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence,
    collate_items, normalize_question,
)
from tinycenn_lm.decision_training import require_finite

SEED = 2034
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}
LAYER = 15 #@param {type:"integer"}
TARGET_LAYERS = (LAYER,)
TRAIN_CASES = 80 #@param {type:"integer"}
DEV_CASES = 160 #@param {type:"integer"}
MAX_LEN = 512 #@param {type:"integer"}
BATCH = 3 #@param {type:"integer"}
STEPS = 400 #@param {type:"integer"}
REFINE_STEPS = 1200 #@param {type:"integer"}
REFINE_TOP_K = 2 #@param {type:"integer"}
TRAIN_CACHE_BATCHES = 1 #@param {type:"integer"}
PROBE_CACHE_BATCHES = 3 #@param {type:"integer"}
AUDIT_CACHE_BATCHES = 3 #@param {type:"integer"}
CHECK_EVERY = 100 #@param {type:"integer"}
OUTPUT_DIR = Path("/content/Laya_V34_AllFull_SparseRetrieval")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DEV_CASES >= 4 and TRAIN_CACHE_BATCHES > 0
assert REFINE_TOP_K >= 1 and TRAIN_CACHE_BATCHES > 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError("Use a CUDA GPU runtime in Google Colab.")
torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda")
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16

source_dir = Path(snapshot_download(
    SOURCE_MODEL, allow_patterns=[
        "rl_agent_config.json", "model.safetensors", "encoder/*", "tokenizer/*",
    ],
))
teacher, source_cfg = build_checkpoint_model(source_dir)
student_eval, _ = build_checkpoint_model(source_dir)
tokenizer = AutoTokenizer.from_pretrained(str(source_dir / "tokenizer"))
teacher.to(device).eval().requires_grad_(False)
student_eval.to(device).eval().requires_grad_(False)
teacher.encoder.config.reference_compile = False
student_eval.encoder.config.reference_compile = False

kinds = layer_kinds(teacher.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
assert LAYER in full_layers, (LAYER, full_layers)
print("GPU:", torch.cuda.get_device_name(0), "AMP:", amp_dtype)
print("Source:", SOURCE_MODEL)
print("Target:", LAYER, "all full attention layers:", full_layers)
#@title V3.4 query routing controls
BLOCK_SIZE = 16 #@param {type:"integer"}
QUERY_CHUNK = 32 #@param {type:"integer"}
ROUTE_BLOCK_COUNTS = (4,8)
SPARSE_MODES = ("local256_remote8","global_top32")
AUDIT_DECISIONS = 120 #@param {type:"integer"}
CUMULATIVE_E2E_DECISIONS = 120 #@param {type:"integer"}
RUN_ALL_FULL_LAYERS = True #@param {type:"boolean"}
# Native full-attention SDPA is the only ground truth.
assert BLOCK_SIZE>=4 and QUERY_CHUNK>=1
print("V3.4 sparse pretrained routing; training updates: 0; official test used: NO")
print("Discovered full-attention layers:",full_layers)


In [ ]:
#@title 3. Leak-free train/dev data and teacher I/O helpers
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
rng = random.Random(SEED)
rng.shuffle(rows)
assert TRAIN_CASES + DEV_CASES <= len(rows)
train_rows = rows[:TRAIN_CASES]
dev_rows = rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, True)
# Split at the CASE level before creating multiple question items per case.
probe_rows = dev_rows[:len(dev_rows) // 2]
audit_rows = dev_rows[len(dev_rows) // 2:]
probe_items = make_items_from_rows(probe_rows, SEED + 1, True)
audit_items = make_items_from_rows(audit_rows, SEED + 2, True)
dev_items = probe_items # backward-compatible alias for training helper use
assert train_items and probe_items and audit_items
assert {str(r["id"]) for r in probe_rows}.isdisjoint({str(r["id"]) for r in audit_rows})

def direct_batch(selected):
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k: v.to(device) for k, v in b.items()}

def batch(items, offset, n=BATCH):
    selected = [items[(offset + i) % len(items)] for i in range(n)]
    return direct_batch(selected)

def _snapshot_attention_argument(value):
    """Retain the exact forward inputs for ModernBERT SDPA replay/timing."""
    if torch.is_tensor(value):
        return value.detach()
    if isinstance(value, tuple):
        return tuple(_snapshot_attention_argument(v) for v in value)
    if isinstance(value, list):
        return [_snapshot_attention_argument(v) for v in value]
    if isinstance(value, dict):
        return {k: _snapshot_attention_argument(v) for k, v in value.items()}
    return value

def _attention_output_tensor(result):
    """ModernBERT variants return either tensor or (tensor, optional weights)."""
    return result[0] if isinstance(result, (tuple, list)) else result

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        # The encoder passes more than x/position_embeddings/attention_mask:
        # position_ids, sliding_window_mask and optional flags are required by
        # transformers 4.57.6 SDPA. Never infer these for benchmarking.
        got["call_args"] = _snapshot_attention_argument(tuple(args))
        got["call_kwargs"] = _snapshot_attention_argument(dict(kwargs))
        got["x"] = args[0].detach()
        got["pos"] = _snapshot_attention_argument(kwargs.get("position_embeddings"))
        got["mask"] = _snapshot_attention_argument(kwargs.get("attention_mask"))
    def post(mod, args, kwargs, out):
        got["y"] = _attention_output_tensor(out).detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove()
        h2.remove()
    return got

def cache_teacher_io(layer_idx, items, batches, offset=0):
    cache = []
    for i in range(batches):
        b = batch(items, offset + i * BATCH, BATCH)
        got = attn_io(teacher, layer_idx, b)
        cache.append({
            "x": got["x"],
            "pos": got["pos"],
            "mask": got["mask"],
            "call_args": got["call_args"],
            "call_kwargs": got["call_kwargs"],
            "y": got["y"],
            "valid": b["attention_mask"].bool(),
        })
    return cache

def local_metrics_from_cache(repl, cache):
    repl.eval()
    diff_sum = power_sum = cos_sum = 0.0
    tokens = 0
    with torch.no_grad():
        for entry in cache:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                pred, _ = repl(
                    entry["x"],
                    position_ids=entry["call_kwargs"].get("position_ids"),
                    attention_mask=entry["mask"],
                )
            require_finite(pred, "V3.4 local probe")
            valid = entry["valid"]
            m = valid[:, :, None].float()
            d = pred.shape[-1]
            diff_sum += float((((pred.float() - entry["y"].float()) ** 2) * m).sum())
            power_sum += float(((entry["y"].float() ** 2) * m).sum())
            count = int(valid.sum())
            cos_sum += float(
                F.cosine_similarity(
                    pred.float()[valid],
                    entry["y"].float()[valid],
                    dim=-1,
                ).sum()
            )
            tokens += count
    nmse = diff_sum / max(power_sum, 1e-8)
    cosine = cos_sum / max(tokens, 1)
    score = nmse + 1.10 * (1.0 - cosine)
    return {"nmse": nmse, "cosine": cosine, "score": score, "tokens": tokens}

print("train/probe/audit decisions:", len(train_items), len(probe_items), len(audit_items))
print("unique train/probe/audit cases:", len(train_rows), len(probe_rows), len(audit_rows))
assert len(probe_items) >= PROBE_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce PROBE_CACHE_BATCHES"
assert len(audit_items) >= AUDIT_CACHE_BATCHES * BATCH, "Increase DEV_CASES or reduce AUDIT_CACHE_BATCHES"
print("official test remains untouched")


In [ ]:
#@title 4. Real query-to-block routed sparse Q/K retrieval (no full QK matrix)
from transformers.models.modernbert.modeling_modernbert import apply_rotary_pos_emb
from tinycenn_lm.laya_lab.core import _valid_tokens

def sparse_route_core(q,k,v,valid,attention_mask=None,
                      mode="local256_remote8",block=BLOCK_SIZE,
                      route_blocks=4,query_chunk=QUERY_CHUNK,
                      return_indices=False):
    """Select tokens by multi-prototype block screening and shortlisted exact QK.

    Does NOT compute NxN QK anywhere. Query chunks limit gathered K/V memory.
    q, k, v: [B, H, T, Dh], with native RoPE already applied to q and k.
    valid: [B,T] boolean; mask: [B,1,T,T] bool/additive or [B,T].
    """
    B,H,T,D=q.shape
    if mode not in ("local256_remote8","local256_remote16","global_top32"):
        raise ValueError(mode)
    if T<1: raise ValueError("Empty sequence is unsupported")
    G=(T+block-1)//block
    extra=G*block-T
    kp=F.pad(k,(0,0,0,extra)).reshape(B,H,G,block,D).float()
    vp=F.pad(valid,(0,extra),value=False).reshape(B,G,block)
    mask=vp[:,None,:,:,None].float()
    representative_mean=(kp*mask).sum(-2)/mask.sum(-2).clamp_min(1)
    first=kp[:,:,:,0,:]
    valid_count=vp.sum(-1).clamp_min(1)
    center=((valid_count-1)//2)[:,None,:].expand(B,H,G)
    last=(valid_count-1)[:,None,:].expand(B,H,G)
    norms=kp.square().sum(-1).masked_fill(~vp[:,None,:,:],-1e30)
    maximal=norms.argmax(-1)
    def get_representative(which):
        return kp.gather(
            3,which[...,None,None].expand(B,H,G,1,D)
        ).squeeze(3)
    prototypes=torch.stack((representative_mean,first,
                            get_representative(center),
                            get_representative(last),
                            get_representative(maximal)),dim=-2)
    group_valid=vp.any(-1)
    block_start=torch.arange(G,device=q.device)*block
    block_end=torch.minimum(block_start+block-1,
                            torch.tensor(T-1,device=q.device))
    ib=torch.arange(B,device=q.device)[:,None,None,None]
    ih=torch.arange(H,device=q.device)[None,:,None,None]
    chunks=[]
    selected_chunks=[]
    available_chunks=[]
    for start in range(0,T,query_chunk):
        stop=min(T,start+query_chunk)
        C=stop-start
        qc=q[:,:,start:stop].float()
        query_pos=torch.arange(start,stop,device=q.device)
        # Block-ranking cost O(B H T G 5 d); no T-by-T query/key matrix.
        # CUDA autocast implicitly casts einsum to FP16, even when both
        # operands were .float(). FP16 cannot represent the -1e9 mask
        # sentinel and would crash in masked_fill. Keep routing logits FP32.
        with torch.autocast(device_type=q.device.type, enabled=False):
            routed=torch.einsum(
                "bhcd,bhgrd->bhcgr",qc.float(),prototypes.float()
            ).amax(-1)/math.sqrt(D)
        assert routed.dtype==torch.float32
        routed=routed.masked_fill(~group_valid[:,None,None,:],-1e9)
        if mode.startswith("local"):
            far_block=((block_start[None,:]<query_pos[:,None]-127) |
                       (block_end[None,:]>query_pos[:,None]+128))
            routed=routed.masked_fill(~far_block[None,None,:,:],-1e9)
        selected_blocks=torch.topk(routed,k=min(route_blocks,G),dim=-1).indices
        candidate=(selected_blocks[...,None]*block+
                   torch.arange(block,device=q.device)).reshape(B,H,C,-1)
        candidate_in_range=candidate<T
        candidate=candidate.clamp(max=T-1)

        def exact_scores(indices,valid_range=None):
            # Advanced indexing returns ONLY the shortlisted B,H,C,N,D tensors.
            chosen_k=k[ib,ih,indices,:].float()
            chosen_v=v[ib,ih,indices,:].float()
            # Elementwise FP32 scores remain FP32 under CUDA autocast.
            # A future matmul/einsum optimization must disable autocast here.
            scores=(qc[:,:,:,None,:]*chosen_k).sum(-1).float()/math.sqrt(D)
            allowed=valid[:,None,None,:].expand(B,H,C,T).gather(-1,indices)
            if valid_range is not None:
                allowed=allowed & valid_range
            if attention_mask is not None:
                if attention_mask.ndim==4:
                    m4=attention_mask[:,:,start:stop,:]
                    m4=m4.expand(B,H,C,T).gather(-1,indices)
                    if m4.dtype==torch.bool:
                        allowed=allowed & m4
                    else:
                        scores=scores+m4.float()
                        allowed=allowed & (m4>-1e4)
                elif attention_mask.ndim!=2:
                    raise ValueError(f"Unsupported mask dimensionality {attention_mask.ndim}")
            return scores.masked_fill(~allowed,-1e9),allowed,chosen_v

        candidate_scores,candidate_valid,_=exact_scores(candidate,candidate_in_range)
        if mode=="global_top32":
            kept=min(32,candidate.shape[-1])
        else:
            kept=min(8 if mode=="local256_remote8" else 16,candidate.shape[-1])
            truly_remote=((candidate<query_pos[None,None,:,None]-127) |
                          (candidate>query_pos[None,None,:,None]+128))
            candidate_valid=candidate_valid & truly_remote
            candidate_scores=candidate_scores.masked_fill(~candidate_valid,-1e9)
        selected=torch.topk(candidate_scores,k=kept,dim=-1).indices
        distant=candidate.gather(-1,selected)
        distant_valid=candidate_valid.gather(-1,selected)
        if mode.startswith("local"):
            offsets=torch.arange(-127,129,device=q.device)
            local_positions=query_pos[:,None]+offsets[None,:]
            local_ok=(local_positions>=0)&(local_positions<T)
            local_positions=local_positions.clamp(0,T-1)
            near=local_positions[None,None,:,:].expand(B,H,C,256)
            indices=torch.cat((near,distant),dim=-1)
            available=torch.cat(
                (local_ok[None,None,:,:].expand(B,H,C,256),distant_valid),
                dim=-1
            )
        else:
            indices=distant
            available=distant_valid
        logits,original_valid,values=exact_scores(indices,available)
        available=available & original_valid
        logits=logits.masked_fill(~available,-1e9)
        prob=torch.softmax(logits,dim=-1)
        mixed=(prob.unsqueeze(-1)*values).sum(-2).to(q.dtype)
        mixed=mixed*valid[:,None,start:stop,None].to(mixed.dtype)
        chunks.append(mixed)
        if return_indices:
            selected_chunks.append(indices)
            available_chunks.append(available)
    output=torch.cat(chunks,dim=2)
    if return_indices:
        return output,torch.cat(selected_chunks,dim=2),torch.cat(available_chunks,dim=2)
    return output

class QueryRoutedFullAttention(nn.Module):
    """Drop-in ModernBERT full-attention module using frozen pretrained QKV."""
    def __init__(self,native,mode="local256_remote8",route_blocks=8,
                 block=BLOCK_SIZE,chunk=QUERY_CHUNK):
        super().__init__()
        self.config=native.config
        self.layer_idx=int(-1 if getattr(native,"layer_idx",None) is None else native.layer_idx)
        self.mode=mode
        self.route_blocks=int(route_blocks)
        self.block=int(block)
        self.chunk=int(chunk)
        self.num_heads=int(native.num_heads)
        self.head_dim=int(native.head_dim)
        self.hidden_size=int(native.config.hidden_size)
        self.Wqkv=copy.deepcopy(native.Wqkv).requires_grad_(False)
        self.Wo=copy.deepcopy(native.Wo).requires_grad_(False)
        self.rotary_emb=copy.deepcopy(native.rotary_emb).requires_grad_(False)
        self.out_drop=copy.deepcopy(native.out_drop).requires_grad_(False)
        self.eval()

    def forward(self,hidden_states,attention_mask=None,position_ids=None,
                position_embeddings=None,sliding_window_mask=None,**kwargs):
        B,T,D=hidden_states.shape
        if position_ids is None:
            position_ids=torch.arange(T,device=hidden_states.device)[None,:]
        qkv=self.Wqkv(hidden_states).view(B,T,3,self.num_heads,self.head_dim)
        cos,sin=self.rotary_emb(qkv,position_ids=position_ids)
        q,k,v=qkv.transpose(3,1).unbind(dim=2)
        q,k=apply_rotary_pos_emb(q,k,cos,sin)
        valid=_valid_tokens(attention_mask,hidden_states)
        core=sparse_route_core(q,k,v,valid,attention_mask=attention_mask,
                               mode=self.mode,block=self.block,
                               route_blocks=self.route_blocks,
                               query_chunk=self.chunk)
        result=self.Wo(core.transpose(1,2).contiguous().reshape(B,T,D))
        return self.out_drop(result),None

    def config_dict(self):
        return dict(architecture="v34_query_routed_sparse",mode=self.mode,
                    route_blocks=self.route_blocks,block=self.block,
                    query_chunk=self.chunk,layer_idx=self.layer_idx,
                    pretrained_qkv=True,pretrained_rope=True,
                    attention_free=False,uses_full_qk_inference=False)

STRATEGIES=[dict(mode=mode,route_blocks=blocks)
            for mode in SPARSE_MODES for blocks in ROUTE_BLOCK_COUNTS]
print("Screen candidates:",STRATEGIES)


In [ ]:
#@title 5. Exact mathematical sparse parity + masking preflight (before all-layer evaluation)
# Synthetic parity: if all keys are retrieved and retained, sparse QKV must
# agree numerically with standard F.scaled_dot_product_attention.
for t in (9,27,32):
    torch.manual_seed(400+t)
    q=torch.randn(2,3,t,8,device=device)
    k=torch.randn_like(q)
    v=torch.randn_like(q)
    valid=torch.ones(2,t,dtype=torch.bool,device=device)
    valid[1,max(1,t-4):]=False
    mask_bool=valid[:,None,None,:].expand(2,1,t,t)
    mask_add=torch.where(mask_bool,0.,-10000.)
    for current_mask in (mask_bool,mask_add):
        sparse=sparse_route_core(q,k,v,valid,current_mask,
                                mode="global_top32",route_blocks=8)
        exact=F.scaled_dot_product_attention(q,k,v,attn_mask=current_mask)
        exact=exact*valid[:,None,:,None]
        diff=float((sparse-exact).abs().amax())
        assert diff<2e-5,(t,current_mask.dtype,diff)
        print(f"Synthetic full-key parity T={t} mask={str(current_mask.dtype)} PASS maxΔ={diff:.7g}")
# Crucial regression: the synthetic test above uses FP32, but a real
# Colab T4 uses torch.float16 autocast. Verify the previously crashing route
# with padded blocks and a long sequence under the actual AMP dtype.
with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
    T_AMP=320
    torch.manual_seed(2034)
    q_amp=torch.randn(1,2,T_AMP,16,device=device,dtype=amp_dtype)
    k_amp=torch.randn_like(q_amp)
    v_amp=torch.randn_like(q_amp)
    valid_amp=torch.arange(T_AMP,device=device)[None,:]<279
    route_amp=sparse_route_core(
        q_amp,k_amp,v_amp,valid_amp,
        attention_mask=valid_amp.long(),mode="local256_remote8",
        route_blocks=8,query_chunk=QUERY_CHUNK
    )
    assert route_amp.shape==q_amp.shape
    assert bool(torch.isfinite(route_amp).all())
    assert bool((route_amp[:,:,279:,:]==0).all())
print("Real AMP long-sequence sparse routing PASS |",amp_dtype,
      "| T=320 with 41 padded tokens")

# Real no-padding-leak check with identical shapes, arbitrary padded content.
native=teacher.encoder.layers[LAYER].attn
repl=QueryRoutedFullAttention(native,mode="local256_remote8",route_blocks=8).to(device)
torch.manual_seed(84)
test_len=279
padded_len=320
x=torch.randn(1,padded_len,repl.hidden_size,device=device)
zero=x.clone()
noise=x.clone()
zero[:,test_len:]=0
noise[:,test_len:]=torch.randn_like(noise[:,test_len:])*7
m=torch.zeros(1,padded_len,dtype=torch.long,device=device)
m[:,:test_len]=1
with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
    a=repl(zero,attention_mask=m)[0][:,:test_len]
    b=repl(noise,attention_mask=m)[0][:,:test_len]
padding_diff=float((a.float()-b.float()).abs().max())
assert padding_diff<1e-3,("Masked padding leak",padding_diff)
print("Real router same-shape masked padding PASS maxΔ",padding_diff)
# Replay native original with EXACT captured kwargs to protect SDPA API parity.
preflight_batch=batch(probe_items,0,BATCH)
captured=attn_io(teacher,LAYER,preflight_batch)
with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
    replay=_attention_output_tensor(native(*captured["call_args"],**captured["call_kwargs"]))
teacher_delta=float((replay.float()-captured["y"].float()).abs().max())
assert teacher_delta<1e-2,teacher_delta
print("Pretrained ModernBERT native SDPA replay PASS maxΔ",teacher_delta)
del repl
torch.cuda.empty_cache()


## Screening: per-layer isolated fidelity before cumulative replacement

The "best" strategy is **chosen independently for each full-attention layer** using case-disjoint PROBE samples. It uses original frozen teacher layer inputs and output, not outputs from already modified upstream layers. Later all-layer execution explicitly measures cumulative distribution shift.

For the audit we keep a separate case-disjoint partition. We use latency both for **native individual attention** and **actual replacement**. **No oracle teacher QK matrix is read during the replacement forward pass**. A separate diagnostic may compute teacher QK on a probe batch solely to measure how much true attention mass the router recovered.

We do not train QKV, rerun VAE, or alter sliding-attention layers. Zero-shot routing may be substantially less accurate than V3.3's oracle. If it fails, the report should motivate training an efficient query-to-block router or using a native fallback rather than pretending a mathematical oracle has been implemented.


In [ ]:
#@title 6. E2E metrics, GPU timings, and per-layer local fidelity
@torch.inference_mode()
def evaluate_logits(student,items,limit=AUDIT_DECISIONS):
    n=agree=teach_gold=stud_gold=0
    js_total=0.0
    for off in range(0,min(limit,len(items)),BATCH):
        chosen=items[off:min(off+BATCH,min(limit,len(items)))]
        b=direct_batch(chosen)
        with torch.autocast(device_type="cuda",dtype=amp_dtype):
            teacher_logits,_=teacher(**b)
            student_logits,_=student(**b)
        allowed=b["marker_mask"].bool()
        pt=torch.softmax(teacher_logits.float().masked_fill(~allowed,-1e9),dim=-1)
        ps=torch.softmax(student_logits.float().masked_fill(~allowed,-1e9),dim=-1)
        it=pt.argmax(-1)
        is_=ps.argmax(-1)
        gold=torch.tensor([v["gold_index"] for v in chosen],device=device)
        n+=len(chosen)
        agree+=int((it==is_).sum())
        teach_gold+=int((it==gold).sum())
        stud_gold+=int((is_==gold).sum())
        avg=(pt+ps)*0.5
        ptl=pt.clamp_min(1e-8);psl=ps.clamp_min(1e-8);ml=avg.clamp_min(1e-8)
        js=0.5*((pt*(ptl.log()-ml.log())).sum(-1)+
                (ps*(psl.log()-ml.log())).sum(-1))
        js_total+=float(js.sum())
    return dict(decisions=n,agreement=agree/max(n,1),
                teacher_gold_accuracy=teach_gold/max(n,1),
                student_gold_accuracy=stud_gold/max(n,1),
                accuracy_drop=(teach_gold-stud_gold)/max(n,1),
                mean_js=js_total/max(n,1))

@torch.inference_mode()
def latency_ms(fn,warmup=3,repeats=8):
    with torch.autocast(device_type="cuda",dtype=amp_dtype):
        for _ in range(warmup):fn()
        torch.cuda.synchronize()
        samples=[]
        for _ in range(repeats):
            started=time.perf_counter()
            fn()
            torch.cuda.synchronize()
            samples.append((time.perf_counter()-started)*1000.0)
    return float(np.median(samples))

def build_route(original,strategy):
    return QueryRoutedFullAttention(
        original,mode=strategy["mode"],route_blocks=strategy["route_blocks"],
        block=BLOCK_SIZE,chunk=QUERY_CHUNK
    ).to(device).eval().requires_grad_(False)

def score(metrics):
    return metrics["nmse"]+1.1*(1.0-metrics["cosine"])

def isolated_probe(idx,strategy,cache):
    with torch.inference_mode():
        layer=build_route(teacher.encoder.layers[idx].attn,strategy)
        quality=local_metrics_from_cache(layer,cache)
        del layer
        return quality

def native_local_report(idx,cache):
    # Original teacher replay is an exact reconstruction control.
    teacher_module=teacher.encoder.layers[idx].attn
    with torch.inference_mode(),torch.autocast(device_type="cuda",dtype=amp_dtype):
        entry=cache[0]
        out=_attention_output_tensor(teacher_module(
            *entry["call_args"],**entry["call_kwargs"]
        ))
    valid=entry["valid"]
    return dict(max_abs_error=float((out.float()[valid]-entry["y"].float()[valid]).abs().max()))

def cache_stat(items):
    sizes=np.asarray([len(x["ids"]) for x in items])
    return dict(items=len(items),median=float(np.median(sizes)),
                p90=float(np.percentile(sizes,90)),max=int(sizes.max()),
                ge256=int((sizes>=256).sum()))


@torch.inference_mode()
def diagnostic_router_recall(idx,strategy,entry):
    """PROBE-ONLY diagnostic: teacher full QK is used for labels and recall,
    NEVER in sparse_route_core inference.
    """
    native=teacher.encoder.layers[idx].attn
    x=entry["x"]
    B,T,D=x.shape
    H,hd=native.num_heads,native.head_dim
    with torch.autocast(device_type="cuda",dtype=amp_dtype):
        qkv=native.Wqkv(x).view(B,T,3,H,hd)
        cos,sin=native.rotary_emb(qkv,
            position_ids=entry["call_kwargs"].get("position_ids"))
        q,k,v=qkv.transpose(3,1).unbind(dim=2)
        q,k=apply_rotary_pos_emb(q,k,cos,sin)
    valid=entry["valid"]
    _output,indices,available=sparse_route_core(
        q,k,v,valid,entry["mask"],mode=strategy["mode"],
        route_blocks=strategy["route_blocks"],block=BLOCK_SIZE,
        query_chunk=QUERY_CHUNK,return_indices=True
    )
    logits=(q.float()@k.float().transpose(-2,-1))/math.sqrt(hd)
    mask=entry["mask"]
    if mask is not None:
        if mask.ndim==4:
            if mask.dtype==torch.bool:
                logits=logits.masked_fill(~mask,-1e9)
            else:
                logits=logits+mask.float()
        elif mask.ndim!=2:
            raise ValueError("Unexpected mask dimensionality")
    logits=logits.masked_fill(~valid[:,None,None,:],-1e9)
    weights=torch.softmax(logits,dim=-1)
    captured=(weights.gather(-1,indices)*available).sum(-1)
    query_mask=valid[:,None,:].expand(B,H,T)
    full_top=weights.topk(k=min(8,T),dim=-1).indices
    hits=(full_top[...,None]==indices[:,:,:,None,:])&available[:,:,:,None,:]
    fraction_top8=hits.any(-1).float().mean(-1)
    # Which of the true high-weight *remote* tokens did this strategy find?
    query_pos=torch.arange(T,device=x.device)[None,None,:,None]
    top_remote=(((full_top < query_pos-127) |
                 (full_top > query_pos+128))).float()
    true_remote=top_remote.sum(-1)
    covered_remote=(hits.any(-1).float()*top_remote).sum(-1)
    far_recall=covered_remote/true_remote.clamp_min(1)
    remote_active=query_mask & (true_remote>0)
    return dict(
        teacher_mass_captured=float(captured[query_mask].mean()),
        fraction_oracle_top8_keys_retrieved=float(fraction_top8[query_mask].mean()),
        fraction_oracle_remote_top8_retrieved=float(
            far_recall[remote_active].mean()) if bool(remote_active.any()) else 1.0,
        queries_with_remote_top8=int(remote_active.sum())
    )

print("Probe lengths:",cache_stat(probe_items))
print("Audit lengths:",cache_stat(audit_items))
print("Untouched native baseline:",evaluate_logits(student_eval,audit_items,limit=AUDIT_DECISIONS))


In [ ]:
#@title 7. Choose each full-attention layer's route using PROBE only
per_layer={}
for idx in full_layers:
    print("\n"+"="*86)
    print(f"LAYER {idx}: screen actual pretrained query-routed sparse replacements")
    print("="*86)
    probe_cache=cache_teacher_io(idx,probe_items,PROBE_CACHE_BATCHES)
    control=native_local_report(idx,probe_cache)
    assert control["max_abs_error"]<1e-2,(idx,control)
    scored=[]
    for conf in STRATEGIES:
        quality=isolated_probe(idx,conf,probe_cache)
        scored.append(dict(configuration=conf,
                           probe=quality,selection_score=score(quality)))
        print(f"  {conf['mode']:<21} blocks={conf['route_blocks']:<2} "
              f"cos={quality['cosine']:.5f} NMSE={quality['nmse']:.5f}")
    scored.sort(key=lambda d:(d["selection_score"],-d["probe"]["cosine"]))
    winning=scored[0]
    # Post-selection explanatory test on the SAME probe batch. Not used
    # to choose between configurations and never used at inference.
    recall=diagnostic_router_recall(idx,winning["configuration"],probe_cache[0])
    per_layer[str(idx)]=dict(
        index=idx,native_replay=control,screen=scored,
        selected=winning["configuration"],selected_probe=winning["probe"],
        selected_probe_router_recall=recall
    )
    print("PROBE-SELECTED:",winning["configuration"])
    print("True teacher attention mass captured:",round(recall["teacher_mass_captured"],4),
          "| true teacher top-8 retrieval:",round(recall["fraction_oracle_top8_keys_retrieved"],4))
    del probe_cache
    torch.cuda.empty_cache()
print("\nProbe selected configurations by full attention layer:")
for name,row in per_layer.items():
    print(name,row["selected"],"probe cosine",row["selected_probe"]["cosine"])
selection_path=OUTPUT_DIR/"v34_probe_selected_per_layer.json"
selection_path.write_text(json.dumps(per_layer,indent=2),encoding="utf-8")
print("Saved selections:",selection_path)


In [ ]:
#@title 8. Held-out isolated layer audit and matched teacher/sparse layer GPU latency
for idx in full_layers:
    conf=per_layer[str(idx)]["selected"]
    audit_cache=cache_teacher_io(idx,audit_items,AUDIT_CACHE_BATCHES)
    layer=build_route(teacher.encoder.layers[idx].attn,conf)
    audit=local_metrics_from_cache(layer,audit_cache)
    entry=audit_cache[0]
    native=teacher.encoder.layers[idx].attn
    native_ms=latency_ms(
        lambda: _attention_output_tensor(
            native(*entry["call_args"],**entry["call_kwargs"])
        ),warmup=3,repeats=10
    )
    sparse_ms=latency_ms(lambda:layer(
        entry["x"],attention_mask=entry["mask"],
        position_ids=entry["call_kwargs"].get("position_ids")
    ),warmup=3,repeats=10)
    per_layer[str(idx)]["isolated_audit"]=audit
    per_layer[str(idx)]["native_attention_ms"]=native_ms
    per_layer[str(idx)]["sparse_attention_ms"]=sparse_ms
    per_layer[str(idx)]["layer_speedup"]=native_ms/max(sparse_ms,1e-8)
    print(f"Layer {idx}: held-out cosine={audit['cosine']:.4f} NMSE={audit['nmse']:.4f} "
          f"native={native_ms:.3f}ms sparse={sparse_ms:.3f}ms "
          f"speedup={native_ms/max(sparse_ms,1e-8):.3f}x")
    del layer,audit_cache
    torch.cuda.empty_cache()


In [ ]:
#@title 9. Install ALL selected sparse replacements; evaluate cumulative decision drift
# Original sliding-attention modules stay byte-for-byte unchanged.
# RUN_ALL_FULL_LAYERS=True by default as requested.
installed=[]
if RUN_ALL_FULL_LAYERS:
    originals={idx:student_eval.encoder.layers[idx].attn for idx in full_layers}
    try:
        for idx in full_layers:
            conf=per_layer[str(idx)]["selected"]
            replacement=build_route(teacher.encoder.layers[idx].attn,conf)
            student_eval.encoder.layers[idx].attn=replacement
            quality=evaluate_logits(student_eval,audit_items,
                                    limit=CUMULATIVE_E2E_DECISIONS)
            installed.append(dict(layer=idx,configuration=conf,metrics=quality))
            print(f"INSTALLED through full layer {idx}: "
                  f"agreement={quality['agreement']:.4f}, "
                  f"gold={quality['student_gold_accuracy']:.4f}, "
                  f"JS={quality['mean_js']:.5f}")
        installed_indices=set(full_layers)
        assert installed_indices=={row["layer"] for row in installed}
        full_quality=evaluate_logits(student_eval,audit_items,
                                     limit=AUDIT_DECISIONS)
        timed=batch(audit_items,0,BATCH)
        with torch.inference_mode():
            native_whole=latency_ms(lambda:teacher(**timed),warmup=2,repeats=6)
            sparse_whole=latency_ms(lambda:student_eval(**timed),warmup=2,repeats=6)
        final_speed=dict(native_ms=native_whole,sparse_ms=sparse_whole,
                         speedup=native_whole/max(sparse_whole,1e-8))
        print("\nALL full-attention layers replaced =",len(full_layers))
        print("Final full-model:",full_quality)
        print("Whole-model latency:",final_speed)
    finally:
        for idx,original in originals.items():
            student_eval.encoder.layers[idx].attn=original
        student_eval.eval().requires_grad_(False)
        torch.cuda.empty_cache()
else:
    full_quality=None
    final_speed=None
    print("Explicitly disabled cumulative replacement.")


In [ ]:
#@title 10. Verdict, machine-readable report, and per-layer fidelity visualization
# Safe to rerun; input checkpoint is not edited and no external model is pushed.
if not per_layer:
    raise RuntimeError("Run the per-layer probe screen before generating report.")
passes_local=all(row.get("isolated_audit",{}).get("cosine",0)>=0.95
                 for row in per_layer.values())
passes_end_to_end=(full_quality is not None and
                    full_quality["agreement"]>=0.95 and
                    full_quality["accuracy_drop"]<=0.01)
passes_speed=(final_speed is not None and final_speed["speedup"]>=1.0)
passes=bool(passes_local and passes_end_to_end and passes_speed)
report=dict(
    version="3.4-query-routed-sparse-all-full-attention",
    model=SOURCE_MODEL,seed=SEED,official_test_used=False,
    pretrained_qkv_frozen=True,
    native_sliding_attention_preserved=True,
    full_attention_layer_indices=list(full_layers),
    full_attention_layers_replaced=len(installed),
    configurations=STRATEGIES,
    per_layer=per_layer,
    cumulative=installed,
    final_all_full_quality=full_quality,
    whole_model_latency=final_speed,
    gates=dict(all_layer_cosine_gte_095=passes_local,
               teacher_agreement_gte_095_and_small_gold_drop=passes_end_to_end,
               whole_model_not_slower=passes_speed,
               overall_pass=passes),
    limitations=[
        "Original V3.3 0.9916 local256+oracle-top8 is an ORACLE, not V3.4 expected quality.",
        "V3.4 replaces exact oracle selection with heuristic query-to-block routing.",
        "The actual inference NEVER calculates full query/key T-by-T scores.",
        "This is SPARSE ATTENTION, not an attention-free or CeNN architecture.",
        "Torch advanced indexing and chunk loops are not fused kernels; speedup not assumed.",
        "Per-layer probe selection uses independent frozen-teacher activations; simultaneous replacement introduces distribution shift.",
        "Audit cases remain disjoint from probe and training cases; external test split unused.",
        "No model uploaded to Hugging Face without meeting gates and explicit request."
    ]
)
report_path=OUTPUT_DIR/"v34_all_full_attention_query_routed_report.json"
report_path.write_text(json.dumps(report,indent=2,allow_nan=False),encoding="utf-8")
print("Saved",report_path)
print("\nFINAL QUALITY GATES",report["gates"])
if not passes:
    print("NOT READY — examine per-layer cosine, cumulative agreement and routing overhead.")
else:
    print("PROMISING, but repeat across seeds, length grids and real deployment kernels.")
import matplotlib.pyplot as plt
names=list(per_layer.keys())
selected=[per_layer[k].get("isolated_audit",{}).get("cosine",0) for k in names]
plt.plot([int(k) for k in names],selected,"o-",label="Held-out routed layer cosine")
plt.axhline(.95,linestyle="--",label="Target 0.95")
plt.ylim(0,1.05)
plt.xticks([int(k) for k in names])
plt.xlabel("Full-attention layer index")
plt.ylabel("Output cosine")
plt.title("V3.4: All pretrained full-attention layers, isolated fidelity")
plt.legend();plt.grid(alpha=.3);plt.show()
